# Finance Dataset Builder

1. Pull S&P 500 tickers + sectors from Wikipedia
2. List my Personal watchlist by theme
3. Merge and deduplicates both lists
4. Pulls Fundamentals from yfinance for every ticker
5. Inspects ava. columns and decide what we keep 
6. Save final CSV

In [2]:
# Core Libaries
import pandas as pd
import yfinance as yf
import time

#Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded.")

Libraries loaded.


##Step 1 : Pull S&P 500 from wikipedia

In [12]:
# Wikipedia blocks default pandas requests — spoof a browser header to get through
import requests
from io import StringIO

headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"} 
#APIs and websites block automated requests constantly. Adding a User-Agent header is one of the first troubleshooting moves you'll make on any scraping or data pull job.

response = requests.get(
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    headers=headers
)

# Now parse the HTML response pandas already has in memory

sp500_df = pd.read_html(StringIO(response.text))[0]

In [14]:
sp500_df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [13]:
sp500_df.columns.to_list()

['Symbol',
 'Security',
 'GICS Sector',
 'GICS Sub-Industry',
 'Headquarters Location',
 'Date added',
 'CIK',
 'Founded']

In [15]:
#keeping the columns we need
sp500_df = sp500_df[['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']].copy()
sp500_df.columns = ['ticker', 'company', 'sector', 'industry']

sp500_df.head()

,ticker,company,sector,industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


In [16]:
#Add column theme
sp500_df['theme'] = 'S&P 500'
sp500_df.head()

,ticker,company,sector,industry,theme
0,MMM,3M,Industrials,Industrial Conglomerates,S&P 500
1,AOS,A. O. Smith,Industrials,Building Products,S&P 500
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,S&P 500
3,ABBV,AbbVie,Health Care,Biotechnology,S&P 500
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,S&P 500


## Step 2: Personal Watchlist

In [ ]:
WATCHLIST = [
    # AI Infrastructure — Power
    ("VST",      "Vistra",                  "AI Infrastructure - Power",      "Equity"),
    ("GEV",      "GE Vernova",              "AI Infrastructure - Power",      "Equity"),
    ("CEG",      "Constellation Energy",    "AI Infrastructure - Power",      "Equity"),
    ("PWR",      "Quanta Services",         "AI Infrastructure - Power",      "Equity"),
    ("XOM",      "Exxon Mobil",             "AI Infrastructure - Power",      "Equity"),
    # AI Infrastructure — Cooling
    ("VRT",      "Vertiv",                  "AI Infrastructure - Cooling",    "Equity"),
    ("MOD",      "Modine Manufacturing",    "AI Infrastructure - Cooling",    "Equity"),
    ("FIX",      "Comfort Systems",         "AI Infrastructure - Cooling",    "Equity"),
    # AI Infrastructure — Networking
    ("ANET",     "Arista Networks",         "AI Infrastructure - Networking", "Equity"),
    ("CIEN",     "Ciena",                   "AI Infrastructure - Networking", "Equity"),
    ("APH",      "Amphenol",                "AI Infrastructure - Networking", "Equity"),
    ("MRVL",     "Marvell Technology",      "AI Infrastructure - Networking", "Equity"),
    ("AVGO",     "Broadcom",                "AI Infrastructure - Networking", "Equity"),
    # AI Infrastructure — Semiconductor Equipment
    ("ASML",     "ASML",                    "AI Infrastructure - Semi Equip", "Equity"),
    ("AMAT",     "Applied Materials",       "AI Infrastructure - Semi Equip", "Equity"),
    ("LRCX",     "Lam Research",            "AI Infrastructure - Semi Equip", "Equity"),
    ("KLAC",     "KLA Corp",                "AI Infrastructure - Semi Equip", "Equity"),
    ("ARM",      "ARM Holdings",            "AI Infrastructure - Semi Equip", "Equity"),
    # AI Infrastructure — Data Center
    ("EME",      "EMCOR Group",             "AI Infrastructure - Data Ctr",   "Equity"),
    ("ETN",      "Eaton",                   "AI Infrastructure - Data Ctr",   "Equity"),
    ("DELL",     "Dell Technologies",       "AI Infrastructure - Data Ctr",   "Equity"),
    ("SMCI",     "Super Micro Computer",    "AI Infrastructure - Data Ctr",   "Equity"),
    # AI Infrastructure — Memory
    ("MU",       "Micron Technology",       "AI Infrastructure - Memory",     "Equity"),    
    ("WDC",      "Western Digital",         "AI Infrastructure - Memory",     "Equity"),
    # Data Center REITs
    ("EQIX",     "Equinix",                 "Data Center REIT",               "REIT"),
    ("DLR",      "Digital Realty",          "Data Center REIT",               "REIT"),
    ("IREN",     "Iris Energy",             "Data Center REIT",               "Equity"),
    ("APLD",     "Applied Digital",         "Data Center REIT",               "Equity"),
    # AI/Tech Other
    ("PLTR",     "Palantir",                "AI/Tech Other",                  "Equity"),
    # Clean Energy
    ("NXT",      "Nextracker",              "Clean Energy",                   "Equity"),
    ("FSLR",     "First Solar",             "Clean Energy",                   "Equity"),
    ("BE",       "Bloom Energy",            "Clean Energy",                   "Equity"),
    ("NEE",      "NextEra Energy",          "Clean Energy",                   "Equity"),
    ("BEP",      "Brookfield Renewable",    "Clean Energy",                   "Equity"),
    ("AES",      "AES Corp",                "Clean Energy",                   "Equity"),
    ("CCJ",      "Cameco",                  "Clean Energy",                   "Equity"),
    # Nuclear
    ("NNE",      "Nano Nuclear",            "Nuclear",                        "Equity"),
    ("OKLO",     "Oklo",                    "Nuclear",                        "Equity"),
    # Power Utilities
    ("SO",       "Southern Company",        "Power Utilities",                "Equity"),
    ("DUK",      "Duke Energy",             "Power Utilities",                "Equity"),
    ("AEP",      "American Electric Power", "Power Utilities",                "Equity"),
    ("EXC",      "Exelon",                  "Power Utilities",                "Equity"),
    ("NRG",      "NRG Energy",              "Power Utilities",                "Equity"),
    ("PCG",      "PG&E",                    "Power Utilities",                "Equity"),
    ("TALEN",    "Talen Energy",            "Power Utilities",                "Equity"),
    # Fiber & Network Infrastructure
    ("LUMN",     "Lumen Technologies",      "Fiber & Network Infra",          "Equity"),
    ("FYBR",     "Frontier Communications", "Fiber & Network Infra",          "Equity"),
    ("UNIT",     "Uniti Group",             "Fiber & Network Infra",          "Equity"),
    ("CSCO",     "Cisco",                   "Fiber & Network Infra",          "Equity"),
    ("JNPR",     "Juniper Networks",        "Fiber & Network Infra",          "Equity"),
    # Emerging Markets
    ("TSM",      "Taiwan Semiconductor",    "Emerging Markets",               "Equity"),
    ("IBN",      "ICICI Bank",              "Emerging Markets",               "Equity"),
    ("MELI",     "MercadoLibre",            "Emerging Markets",               "Equity"),
    ("SE",       "Sea Limited",             "Emerging Markets",               "Equity"),
    ("BABA",     "Alibaba",                 "Emerging Markets",               "Equity"),
    # EM ETFs
    ("IEMG",     "iShares Core MSCI EM",    "EM ETF",                         "ETF"),
    ("EWY",      "iShares MSCI S.Korea",    "EM ETF",                         "ETF"),
    ("INDA",     "iShares MSCI India",      "EM ETF",                         "ETF"),
    ("EWZ",      "iShares MSCI Brazil",     "EM ETF",                         "ETF"),
    ("VWO",      "Vanguard FTSE EM",        "EM ETF",                         "ETF"),
    # Income / Options ETFs
    ("JEPI",     "JPMorgan Equity Premium", "Income ETF",                     "ETF"),
    ("JEPQ",     "JPMorgan Nasdaq Equity",  "Income ETF",                     "ETF"),
    ("XYLD",     "Global X S&P500 CC",      "Income ETF",                     "ETF"),
    ("QYLD",     "Global X Nasdaq CC",      "Income ETF",                     "ETF"),
    ("YMAX",     "YieldMax Universe",       "Income ETF",                     "ETF"),
    ("YMAG",     "YieldMax Mag7",           "Income ETF",                     "ETF"),
    ("MSFO",     "YieldMax MSFT",           "Income ETF",                     "ETF"),
    ("NVDO",     "YieldMax NVDA",           "Income ETF",                     "ETF"),
    ("TSLY",     "YieldMax TSLA",           "Income ETF",                     "ETF"),
    ("CONY",     "YieldMax COIN",           "Income ETF",                     "ETF"),
    ("AMZY",     "YieldMax AMZN",           "Income ETF",                     "ETF"),
    # Dividend / Mixed
    ("SOFI",     "SoFi Technologies",       "Dividend/Mixed",                 "Equity"),
    ("SPGI",     "S&P Global",              "Dividend/Mixed",                 "Equity"),
    ("ZIM",      "ZIM Integrated Shipping", "Dividend/Mixed",                 "Equity"),
    ("MDLZ",     "Mondelez",                "Dividend/Mixed",                 "Equity"),
    # Fintech
    ("SQ",       "Block",                   "Fintech",                        "Equity"),
    ("AFRM",     "Affirm",                  "Fintech",                        "Equity"),
    ("HOOD",     "Robinhood",               "Fintech",                        "Equity"),
    # Defense
    ("LMT",      "Lockheed Martin",         "Defense",                        "Equity"),
    ("RTX",      "RTX Corp",                "Defense",                        "Equity"),
    # Commodities
    ("GLD",      "SPDR Gold Shares",        "Commodities",                    "ETF"),
    ("SLV",      "iShares Silver Trust",    "Commodities",                    "ETF"),
    ("URNM",     "Sprott Uranium Miners",   "Commodities",                    "ETF"),
    # Small Cap / Speculative
    ("BCLI",     "Brainstorm Cell",         "Small Cap/Speculative",          "Equity"),
    ("KEN",      "Kenon Holdings",          "Small Cap/Speculative",          "Equity"),
    ("WKSP",     "Worksport",               "Small Cap/Speculative",          "Equity"),
    # Crypto Adjacent
    ("COIN",     "Coinbase",                "Crypto Adjacent",                "Equity"),
    ("MSTR",     "MicroStrategy",           "Crypto Adjacent",                "Equity"),
    ("RIOT",     "Riot Platforms",          "Crypto Adjacent",                "Equity"),
    # Crypto
    ("BTC-USD",  "Bitcoin",                 "Crypto",                         "Crypto"),
    ("XRP-USD",  "XRP",                     "Crypto",                         "Crypto"),
    ("XLM-USD",  "Stellar",                 "Crypto",                         "Crypto"),
    ("ALGO-USD", "Algorand",                "Crypto",                         "Crypto"),
    ("HBAR-USD", "Hedera",                  "Crypto",                         "Crypto"),
    ("LINK-USD", "Chainlink",               "Crypto",                         "Crypto"),
    ("XDC-USD",  "XDC Network",             "Crypto",                         "Crypto"),
]

In [ ]:
#convert to dataframe to merge with S&P500 list 
watchlist_df = pd.DataFrame(WATCHLIST, columns=['ticker', 'company', 'theme', 'asset_type'])

print(f"Watchlist Tickers: {len(watchlist_df)}")
print(f"\nBY theme:")
print(watchlist_df.groupby('theme').size().sort_values(ascending=False))

Watchlist Tickers: 96

BY theme:
theme
Income ETF                        11
Clean Energy                       7
Crypto                             7
Power Utilities                    7
AI Infrastructure - Power          5
AI Infrastructure - Networking     5
Emerging Markets                   5
EM ETF                             5
Fiber & Network Infra              5
AI Infrastructure - Semi Equip     5
Data Center REIT                   4
Dividend/Mixed                     4
AI Infrastructure - Data Ctr       4
AI Infrastructure - Cooling        3
Small Cap/Speculative              3
Fintech                            3
Commodities                        3
Crypto Adjacent                    3
AI Infrastructure - Memory         2
Defense                            2
Nuclear                            2
AI/Tech Other                      1
dtype: int64
